# ZeroSyl Boundary Evaluation 2

Scores a ZeroSyl-estimated `TextGrid` against a ground-truth `TextGrid` and shows
everything in **one interactive viewer** that fits on screen:

| Bar | Contents |
|---|---|
| **Top** | Spectrogram of the `.wav` |
| **Middle** | Ground truth: **words**, **phones**, **syllables** (three equal rows, same total height as the prediction bar) |
| **Bottom** | ZeroSyl prediction (with its labels) |

* Blue **tolerance bands** sit around every scored ground-truth syllable boundary and bleed down into the prediction bar.
* Over the prediction bar: **green dot = TP**, **red cross = FP**, **orange triangle = FN**.
* Silences (empty intervals) are a slightly darker shade in both ground truth and prediction.
* Choose **Start / End** (or use the scroll slider) to zoom. **Play** plays exactly the visible section, and a thin black line sweeps across all three bars in sync with the audio.
* **Click a syllable** in the ground truth (or a word / phone) or in the prediction to play just that interval from start to finish; it is highlighted while it plays. Silences are not played. Clicking the spectrogram plays from that point to the end of the visible section.

**Scoring is unchanged:** the functions in section 2 are copied unmodified from `boundaries.py`.

## 1. Configuration

In [ ]:
from pathlib import Path

GROUND_TRUTH_TEXTGRID = Path("id_3_main_post_test_syllables.TextGrid")
ESTIMATED_TEXTGRID = Path("id_3_main_estimated.TextGrid")
AUDIO_WAV = Path("id_3_main_post_test.wav")

# Tier names
GT_WORD_TIER = "words"
GT_PHONE_TIER = "phones"
GT_SYLLABLE_TIER = "syllables"   # this is the tier that is scored (as in boundaries.py)
ESTIMATED_TIER = "syllables"

TOLERANCE_S = 0.05               # 50 ms, same default as boundaries.py

# Viewer
INITIAL_START_S = 0.0            # section shown when the viewer first opens
INITIAL_END_S = 8.0
WIDGET_HEIGHT_PX = None          # None = fit to the browser window height; or set e.g. 760

# Spectrogram
SPEC_WINDOW_S = 0.012            # analysis window
SPEC_HOP_S = 0.004               # hop between frames
SPEC_MAX_FREQ_HZ = 8000
SPEC_DYNAMIC_RANGE_DB = 60

for p in (GROUND_TRUTH_TEXTGRID, ESTIMATED_TEXTGRID, AUDIO_WAV):
    print(("OK      " if p.exists() else "MISSING ") + str(p.resolve()))
print(f"Tolerance: ±{TOLERANCE_S * 1000:.0f} ms")

## 2. `boundaries.py` scoring methodology (copied unmodified)

Only the `.pt` / dev-set loading front end (`evaluate_boundary_metrics`) is left out, because this notebook reads two `TextGrid`s directly instead.

In [ ]:
import itertools
from typing import List, Tuple

import numpy as np
import tgt


def split_utterance(
    seg: List[float], ref: tgt.IntervalTier, tolerance: float
) -> Tuple[List[List[float]], List[List[float]]]:

    ref_out = [
        list(ref_utt)
        for k, ref_utt in itertools.groupby(ref.intervals, lambda x: x.text != "")
        if k
    ]

    seg_out = []
    for ref_utt in ref_out:
        ref_utt_onset = ref_utt[0].start_time + tolerance
        ref_utt_offset = ref_utt[-1].end_time - tolerance
        seg_out.append([s for s in seg if ref_utt_onset < s < ref_utt_offset])
        seg_out[-1].append(ref_utt[-1].end_time)

    return seg_out, [
        [float(interval.end_time) for interval in ref_utt] for ref_utt in ref_out
    ]


def get_p_r_f1(n_seg: int, n_ref: int, n_hit: int) -> Tuple[float, float, float]:

    if n_seg == n_ref == 0:
        return 0, 0, -np.inf
    elif n_hit == 0:
        return 0, 0, 0

    if n_seg != 0:
        precision = float(n_hit / n_seg)
    else:
        precision = np.inf

    if n_ref != 0:
        recall = float(n_hit / n_ref)
    else:
        recall = np.inf

    if precision + recall != 0:
        f1_score = 2 * precision * recall / (precision + recall)
    else:
        f1_score = -np.inf

    return precision, recall, f1_score


def eval_boundaries(
    seg: List[float], ref: List[float], tolerance: float = 0.05
) -> Tuple[float, float, float]:

    n_seg, n_ref, n_hit = 0, 0, 0
    assert len(seg) == len(ref)

    for i_utterance in range(len(seg)):
        prediction = seg[i_utterance]
        ground_truth = ref[i_utterance]

        if (
            len(prediction) > 0
            and len(ground_truth) > 0
            and abs(prediction[-1] - ground_truth[-1]) <= tolerance
        ):
            prediction = prediction[:-1]
            if len(ground_truth) > 0:
                ground_truth = ground_truth[:-1]

        n_seg += len(prediction)
        n_ref += len(ground_truth)

        if len(prediction) == 0 or len(ground_truth) == 0:
            continue

        for i_ref in ground_truth:
            for i, i_seg in enumerate(prediction):
                if abs(i_ref - i_seg) <= tolerance:
                    n_hit += 1
                    prediction.pop(i)
                    break

    return get_p_r_f1(n_seg, n_ref, n_hit)


def get_os(precision: float, recall: float) -> float:

    if precision == 0:
        return -np.inf
    else:
        return recall / precision - 1


def get_rvalue(precision: float, recall: float) -> float:

    os = get_os(precision, recall)
    r1 = np.sqrt((1 - recall) ** 2 + os**2)
    r2 = (-os + recall - 1) / np.sqrt(2)

    return 1 - (np.abs(r1) + np.abs(r2)) / 2


def eval_token_boundaries(
    seg: List[float], ref: List[float], tolerance: float = 0.05
) -> Tuple[float, float, float]:

    n_tokens_seg, n_tokens_ref, n_tokens_hit = 0, 0, 0
    assert len(seg) == len(ref)

    for i_utterance in range(len(seg)):
        prediction = seg[i_utterance]
        ground_truth = ref[i_utterance]

        seg_segments = [(a, b) for a, b in itertools.pairwise([0] + prediction)]
        ref_segments = [(a, b) for a, b in itertools.pairwise([0] + ground_truth)]

        ref_intervals = []
        for word_start, word_end in ref_segments:
            ref_intervals.append(
                (
                    (max(0, word_start - tolerance), word_start + tolerance),
                    (word_end - tolerance, word_end + tolerance),
                )
            )

        n_tokens_ref += len(ref_intervals)
        n_tokens_seg += len(seg_segments)

        for seg_start, seg_end in seg_segments:
            for i_segment, (ref_start_interval, ref_end_interval) in enumerate(
                ref_intervals
            ):
                ref_start_lower, ref_start_upper = ref_start_interval
                ref_end_lower, ref_end_upper = ref_end_interval

                if (
                    ref_start_lower <= seg_start <= ref_start_upper
                    and ref_end_lower <= seg_end <= ref_end_upper
                ):
                    n_tokens_hit += 1
                    ref_intervals.pop(i_segment)
                    break

    return get_p_r_f1(n_tokens_seg, n_tokens_ref, n_tokens_hit)

## 3. Load the `TextGrid`s and score

The estimated boundary list is built exactly as `boundaries.py` builds it from its segments (`starts + last end`), but reading the intervals of the estimated `TextGrid` tier instead of a `.pt` file.

In [ ]:
import pandas as pd
from IPython.display import HTML, display

gt_textgrid = tgt.read_textgrid(GROUND_TRUTH_TEXTGRID, include_empty_intervals=True)
est_textgrid = tgt.read_textgrid(ESTIMATED_TEXTGRID, include_empty_intervals=True)

gt_word_tier = gt_textgrid.get_tier_by_name(GT_WORD_TIER)
gt_phone_tier = gt_textgrid.get_tier_by_name(GT_PHONE_TIER)
gt_tier = gt_textgrid.get_tier_by_name(GT_SYLLABLE_TIER)
est_tier = est_textgrid.get_tier_by_name(ESTIMATED_TIER)

estimated_boundaries = [iv.start_time for iv in est_tier.intervals] + [
    est_tier.intervals[-1].end_time
]
estimated_boundaries = [round(float(b), 4) for b in estimated_boundaries]

seg, ref = split_utterance(estimated_boundaries, gt_tier, tolerance=TOLERANCE_S)

# eval_* functions mutate their inputs, so score from copies
precision, recall, f1 = eval_boundaries(
    [list(s) for s in seg], [list(r) for r in ref], tolerance=TOLERANCE_S
)
os_value = get_os(precision, recall)
rvalue = get_rvalue(precision, recall)
token_precision, token_recall, token_f1 = eval_token_boundaries(
    [list(s) for s in seg], [list(r) for r in ref], tolerance=TOLERANCE_S
)

metrics = pd.DataFrame(
    {
        "metric": [
            "Boundary precision", "Boundary recall", "Boundary F1",
            "Over-segmentation (OS)", "R-value",
            "Token precision", "Token recall", "Token F1",
        ],
        "value": [
            precision, recall, f1, os_value, rvalue,
            token_precision, token_recall, token_f1,
        ],
    }
)
metrics["value"] = metrics["value"].round(4)
print(f"{len(seg)} utterances scored, tolerance ±{TOLERANCE_S * 1000:.0f} ms")
display(metrics)

## 4. Which boundaries were TP / FP / FN (for the viewer)

`eval_boundaries` only returns counts. `eval_boundaries_detailed` runs the same loop with the same matching rules but keeps the matched pairs and the unmatched times so they can be drawn. The assertion confirms it reproduces the scores above exactly.

In [ ]:
def eval_boundaries_detailed(
    seg: List[List[float]], ref: List[List[float]], tolerance: float = 0.05
):
    tp_pairs, fp_times, fn_times = [], [], []
    assert len(seg) == len(ref)

    for i_utterance in range(len(seg)):
        prediction = list(seg[i_utterance])
        ground_truth = list(ref[i_utterance])

        # same utterance-final exclusion as eval_boundaries
        if (
            len(prediction) > 0
            and len(ground_truth) > 0
            and abs(prediction[-1] - ground_truth[-1]) <= tolerance
        ):
            prediction.pop()
            ground_truth.pop()

        if len(prediction) == 0 or len(ground_truth) == 0:
            fn_times.extend(ground_truth)
            fp_times.extend(prediction)
            continue

        remaining_pred = list(prediction)
        hit_flags = [False] * len(ground_truth)
        for idx_ref, i_ref in enumerate(ground_truth):
            for i, i_seg in enumerate(remaining_pred):
                if abs(i_ref - i_seg) <= tolerance:
                    tp_pairs.append((i_ref, i_seg))
                    hit_flags[idx_ref] = True
                    remaining_pred.pop(i)
                    break

        fn_times.extend([t for t, hit in zip(ground_truth, hit_flags) if not hit])
        fp_times.extend(remaining_pred)

    return tp_pairs, fp_times, fn_times


tp_pairs, fp_times, fn_times = eval_boundaries_detailed(seg, ref, tolerance=TOLERANCE_S)

n_hit = len(tp_pairs)
chk_p, chk_r, chk_f1 = get_p_r_f1(n_hit + len(fp_times), n_hit + len(fn_times), n_hit)
assert abs(chk_p - precision) < 1e-9 and abs(chk_r - recall) < 1e-9 and abs(chk_f1 - f1) < 1e-9
print(f"TP={n_hit}  FP={len(fp_times)}  FN={len(fn_times)}  "
      f"(P={chk_p:.4f}, R={chk_r:.4f}, F1={chk_f1:.4f} - identical to the scores above)")

# Ground-truth boundaries that are actually scored (tolerance bands are drawn around these)
scored_gt_boundaries = sorted([r for r, _ in tp_pairs] + list(fn_times))

## 5. Spectrogram

Wide-band spectrogram (grey, loud = dark) rendered once to an image that the viewer crops and stretches as you zoom.

In [ ]:
import base64
import io

import soundfile as sf
from PIL import Image
from scipy.signal import spectrogram

audio_data, sample_rate = sf.read(AUDIO_WAV)
if audio_data.ndim > 1:
    audio_data = audio_data.mean(axis=1)
recording_duration = len(audio_data) / sample_rate

emphasised = np.append(audio_data[0], audio_data[1:] - 0.97 * audio_data[:-1])
nperseg = int(round(SPEC_WINDOW_S * sample_rate))
hop = int(round(SPEC_HOP_S * sample_rate))
nfft = 1 << int(np.ceil(np.log2(nperseg * 2)))

freqs, frame_centres, power = spectrogram(
    emphasised, fs=sample_rate, window="hann", nperseg=nperseg,
    noverlap=nperseg - hop, nfft=nfft, mode="psd",
)
max_freq = min(SPEC_MAX_FREQ_HZ, sample_rate / 2)
power = power[freqs <= max_freq]
db = 10 * np.log10(power + 1e-12)
db_max = np.percentile(db, 99.9)
norm = np.clip((db - (db_max - SPEC_DYNAMIC_RANGE_DB)) / SPEC_DYNAMIC_RANGE_DB, 0, 1)
grey = (255 * (1 - norm)).astype(np.uint8)[::-1]          # low frequencies at the bottom, loud = dark

buffer = io.BytesIO()
Image.fromarray(grey, mode="L").save(buffer, format="JPEG", quality=90)
spectrogram_uri = "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()

spec_info = dict(
    start=float(frame_centres[0] - SPEC_HOP_S / 2),         # time at the left edge of the first column
    hop=hop / sample_rate,
    nFrames=int(grey.shape[1]),
    maxFreq=float(max_freq),
)
print(f"Spectrogram: {grey.shape[1]} frames x {grey.shape[0]} bins, 0-{max_freq:.0f} Hz, "
      f"{len(buffer.getvalue()) / 1e6:.1f} MB image")

## 6. The viewer

Controls: **Start / End** boxes, the **scroll slider** and **◀ ▶** buttons move or zoom the visible section. **Play** plays exactly that section (tick *Loop* to repeat it). The black line follows the audio across all three bars. **Click any syllable** (GT or prediction) to hear just that syllable; with *Loop* ticked it repeats.

In [ ]:
import json
import uuid

VIEWER_TEMPLATE = r"""
<div id="zsv-__UID__" class="zsv">
<style>
#zsv-__UID__ { font: 12px/1.3 Helvetica, Arial, sans-serif; color:#222; background:#fff; border:1px solid #c8c8c8;
  border-radius:6px; padding:6px 8px; box-sizing:border-box; width:100%; }
#zsv-__UID__ .row { display:flex; flex-wrap:wrap; align-items:center; gap:4px 14px; margin:0 0 5px 0; }
#zsv-__UID__ label { color:#222; }
#zsv-__UID__ input, #zsv-__UID__ button { color:#222; background:#f4f4f4; border:1px solid #a8a8a8; border-radius:4px; font-size:12px; }
#zsv-__UID__ input[type=number] { width:74px; padding:2px 4px; }
#zsv-__UID__ input[type=range] { flex:1 1 240px; min-width:180px; max-width:560px; padding:0; background:transparent; border:0; }
#zsv-__UID__ input[type=checkbox] { vertical-align:middle; }
#zsv-__UID__ button { padding:3px 12px; cursor:pointer; font-size:13px; }
#zsv-__UID__ button.play { min-width:86px; font-weight:600; }
#zsv-__UID__ canvas { display:block; width:100%; cursor:pointer; }
#zsv-__UID__ .grey { color:#666; }
#zsv-__UID__ .tnow { font-variant-numeric:tabular-nums; min-width:90px; display:inline-block; }
</style>
<div class="controls">
  <div class="row">
    <label>Start (s) <input type="number" class="start" step="0.1" min="0"></label>
    <label>End (s) <input type="number" class="end" step="0.1" min="0"></label>
    <button class="prev" title="Shift left by half a window">&#9664;</button>
    <input type="range" class="scroll" min="0" max="1" step="0.01" value="0" title="Scroll through the recording">
    <button class="next" title="Shift right by half a window">&#9654;</button>
    <span class="grey">window <span class="win"></span> s</span>
    <span class="grey">&middot; click a syllable (GT or prediction) to play just that syllable</span>
  </div>
  <div class="row">
    <button class="play">&#9654; Play</button>
    <button class="stop" title="Stop and hide the playhead">&#9632; Stop</button>
    <label><input type="checkbox" class="loop"> Loop</label>
    <span>time <span class="tnow">&ndash;</span></span>
    <span><b style="color:#2CA02C">&#9679;</b> TP &nbsp;<b style="color:#D62728">&#10005;</b> FP &nbsp;<b style="color:#FF7F0E">&#9660;</b> FN
      &nbsp;<span style="display:inline-block;width:14px;height:10px;background:rgba(60,110,200,.28);border:1px solid rgba(40,80,170,.7);vertical-align:middle"></span> tolerance</span>
    <span class="counts"></span>
    <span class="grey overall"></span>
  </div>
</div>
<canvas></canvas>
</div>
<script>
(function(){
const CFG = __PAYLOAD__;
const AUDIO_B64 = "__AUDIO_B64__";
const SPEC_URI = "__SPEC_URI__";

const root = document.getElementById("zsv-__UID__");
const $ = s => root.querySelector(s);
const cv = $("canvas"), ctx = cv.getContext("2d");
const stat = document.createElement("canvas"), sctx = stat.getContext("2d");
const FONT = "Helvetica, Arial, 'DejaVu Sans', sans-serif";

const L = 100, R = 12, GAP = 6, AXH = 24, TOP = 4, MIN_WIN = 0.3;
const dur = CFG.duration, tol = CFG.tolerance;
let vs = CFG.initialStart, ve = Math.min(CFG.initialEnd, dur);
let playing = false, playPos = null, raf = null, geom = null, specReady = false;
let playEnd = ve, loopFrom = vs, sel = null;        // sel = interval currently being played via a click

// ---- audio (decoded from the embedded WAV)
const bin = atob(AUDIO_B64), bytes = new Uint8Array(bin.length);
for (let i = 0; i < bin.length; i++) bytes[i] = bin.charCodeAt(i);
const audio = new Audio(URL.createObjectURL(new Blob([bytes], {type: "audio/wav"})));
audio.preload = "auto";

// ---- spectrogram image
const specImg = new Image();
specImg.onload = () => { specReady = true; drawStatic(); render(); };
specImg.src = SPEC_URI;

const clamp = (x, a, b) => Math.max(a, Math.min(b, x));
const tx = t => L + (t - vs) / (ve - vs) * geom.plotW;

// ------------------------------------------------------------------ layout
function layout() {
  const dpr = window.devicePixelRatio || 1;
  const W = Math.max(600, Math.floor((root.clientWidth || 1000) - 18));
  const ctrlH = $(".controls").offsetHeight;
  const totalH = CFG.heightPx || Math.max(480, Math.min(window.innerHeight - 24, 900));
  const canvasH = Math.max(380, totalH - ctrlH - 20);
  const barH = Math.floor((canvasH - TOP - 2 * GAP - AXH) / 3);
  const H = TOP + 3 * barH + 2 * GAP + AXH;
  for (const c of [cv, stat]) { c.width = Math.round(W * dpr); c.height = Math.round(H * dpr); }
  cv.style.width = W + "px"; cv.style.height = H + "px";
  geom = { W, H, dpr, barH, specY: TOP, gtY: TOP + barH + GAP, predY: TOP + 2 * (barH + GAP),
           axisY: TOP + 3 * barH + 2 * GAP, plotW: W - L - R };
  drawStatic(); render();
}

// ------------------------------------------------------------------ text helpers
function drawLabel(c, text, xa, xb, y, h, fs) {
  if (!text) return;
  xa = Math.max(xa, L); xb = Math.min(xb, L + geom.plotW);
  const w = xb - xa;
  if (w < 3) return;
  c.font = fs + "px " + FONT;
  const tw = c.measureText(text).width;
  const cx = (xa + xb) / 2, cy = y + h / 2;
  if (tw + 4 <= w) { c.fillText(text, cx, cy); return; }
  if (w >= fs + 2) {                                     // not enough width: write it vertically
    let s = text; const maxLen = h - 6;
    if (tw > maxLen) {
      while (s.length > 1 && c.measureText(s + "…").width > maxLen) s = s.slice(0, -1);
      if (s.length < 1) return;
      s += "…";
    }
    c.save(); c.translate(cx, cy); c.rotate(-Math.PI / 2); c.fillText(s, 0, 0); c.restore();
  }
}

function gutterLabel(c, text, y, h) {
  c.save(); c.fillStyle = "#222"; c.font = "bold 12px " + FONT; c.textAlign = "center"; c.textBaseline = "middle";
  c.translate(14, y + h / 2); c.rotate(-Math.PI / 2); c.fillText(text, 0, 0); c.restore();
}

// ------------------------------------------------------------------ static layers
function drawStatic() {
  if (!geom) return;
  const { W, H, dpr, barH, specY, gtY, predY, axisY, plotW } = geom;
  const c = sctx;
  c.setTransform(dpr, 0, 0, dpr, 0, 0);
  c.fillStyle = "#fff"; c.fillRect(0, 0, W, H);
  c.textBaseline = "middle"; c.textAlign = "center";
  const tierH = barH / 3;
  const span = ve - vs;
  const GT_BASE = "#E8EEF7", GT_SIL = "#CCD6E6", PR_BASE = "#E6F3E6", PR_SIL = "#C3DBC3";

  // ---- 1) spectrogram
  c.save(); c.beginPath(); c.rect(L, specY, plotW, barH); c.clip();
  c.fillStyle = "#fff"; c.fillRect(L, specY, plotW, barH);
  if (specReady) {
    const sp = CFG.spec, specEnd = sp.start + sp.nFrames * sp.hop;
    const t0 = Math.max(vs, sp.start), t1 = Math.min(ve, specEnd);
    if (t1 > t0) {
      c.imageSmoothingEnabled = true; c.imageSmoothingQuality = "high";
      c.drawImage(specImg, (t0 - sp.start) / sp.hop, 0, (t1 - t0) / sp.hop, specImg.height,
                  tx(t0), specY, tx(t1) - tx(t0), barH);
    }
  }
  c.restore();

  // ---- 2) fills (silence = empty text = darker shade)
  const fillTier = (ivs, y, h, base, sil) => {
    c.save(); c.beginPath(); c.rect(L, y, plotW, h); c.clip();
    for (const [s, e, t] of ivs) {
      if (e <= vs || s >= ve) continue;
      const xa = tx(Math.max(s, vs)), xb = tx(Math.min(e, ve));
      c.fillStyle = t ? base : sil; c.fillRect(xa, y, xb - xa + 0.5, h);
    }
    c.restore();
  };
  const tiers = [["words", CFG.gt.words], ["phones", CFG.gt.phones], ["syllables", CFG.gt.syllables]];
  tiers.forEach(([n, ivs], k) => fillTier(ivs, gtY + k * tierH, tierH, GT_BASE, GT_SIL));
  fillTier(CFG.est, predY, barH, PR_BASE, PR_SIL);

  // ---- 3) tolerance bands: from the GT syllable row, down through the prediction bar
  const bandTop = gtY + 2 * tierH, bandBot = predY + barH;
  c.save(); c.beginPath(); c.rect(L, 0, plotW, H); c.clip();
  for (const g of CFG.scoredGT) {
    if (g + tol < vs || g - tol > ve) continue;
    const xa = tx(g - tol), xb = tx(g + tol), xc = tx(g);
    c.fillStyle = "rgba(60,110,200,0.24)"; c.fillRect(xa, bandTop, xb - xa, bandBot - bandTop);
    c.strokeStyle = "rgba(50,95,185,0.55)"; c.lineWidth = 1;
    c.beginPath(); c.moveTo(xa, bandTop); c.lineTo(xa, bandBot); c.moveTo(xb, bandTop); c.lineTo(xb, bandBot); c.stroke();
    c.strokeStyle = "rgba(30,70,170,0.95)"; c.setLineDash([4, 3]);
    c.beginPath(); c.moveTo(xc, gtY + 2 * tierH); c.lineTo(xc, bandBot); c.stroke(); c.setLineDash([]);
  }
  c.restore();

  // ---- 4) interval edges + text
  const edges = (ivs, y, h, col) => {
    c.save(); c.beginPath(); c.rect(L, y, plotW, h); c.clip();
    c.strokeStyle = col; c.lineWidth = 1; c.beginPath();
    for (const [s, e] of ivs) {
      if (s > vs && s < ve) { const x = Math.round(tx(s)) + 0.5; c.moveTo(x, y); c.lineTo(x, y + h); }
    }
    c.stroke(); c.restore();
  };
  const texts = (ivs, y, h, fs) => {
    c.save(); c.beginPath(); c.rect(L, y, plotW, h); c.clip();
    c.fillStyle = "#111"; c.textAlign = "center"; c.textBaseline = "middle";
    for (const [s, e, t] of ivs) {
      if (!t || e <= vs || s >= ve) continue;
      drawLabel(c, t, tx(Math.max(s, vs)), tx(Math.min(e, ve)), y, h, fs);
    }
    c.restore();
  };
  const fsGT = clamp(Math.round(tierH * 0.38), 9, 14);
  tiers.forEach(([n, ivs], k) => {
    edges(ivs, gtY + k * tierH, tierH, "#7F93B3");
    texts(ivs, gtY + k * tierH, tierH, fsGT);
  });
  const fsPred = clamp(Math.round(barH * 0.085), 11, 16);
  const labelH = barH * 0.68;                              // top part of the prediction bar = text, bottom = markers
  edges(CFG.est, predY, barH, "#6FA56F");
  texts(CFG.est, predY, labelH, fsPred);

  // ---- 5) frames, separators, gutter labels
  c.strokeStyle = "#555"; c.lineWidth = 1;
  for (const y of [specY, gtY, predY]) c.strokeRect(L + 0.5, y + 0.5, plotW - 1, barH - 1);
  c.strokeStyle = "#555";
  for (let k = 1; k < 3; k++) { c.beginPath(); c.moveTo(L, gtY + k * tierH + 0.5); c.lineTo(L + plotW, gtY + k * tierH + 0.5); c.stroke(); }
  gutterLabel(c, "Spectrogram", specY, barH);
  gutterLabel(c, "Ground truth", gtY, barH);
  gutterLabel(c, "ZeroSyl prediction", predY, barH);
  c.fillStyle = "#555"; c.font = "11px " + FONT; c.textAlign = "right"; c.textBaseline = "middle";
  tiers.forEach(([n], k) => c.fillText(n, L - 6, gtY + k * tierH + tierH / 2));
  const mf = CFG.spec.maxFreq, fmt = f => f === 0 ? "0 Hz" : (f / 1000) + " kHz";
  c.textBaseline = "top";    c.fillText(fmt(mf), L - 6, specY + 2);
  c.textBaseline = "middle"; c.fillText(fmt(mf / 2), L - 6, specY + barH / 2);
  c.textBaseline = "bottom"; c.fillText("0 Hz", L - 6, specY + barH - 2);

  // ---- 6) TP / FP / FN markers over the prediction bar
  const my = predY + barH * 0.84, r = clamp(barH * 0.04, 4, 8);
  c.save(); c.beginPath(); c.rect(L - r - 2, predY, plotW + 2 * r + 4, barH); c.clip();
  for (const [g, e] of CFG.tp) {                           // green dot at the predicted boundary
    if (e < vs || e > ve) continue;
    c.beginPath(); c.arc(tx(e), my, r, 0, 2 * Math.PI);
    c.fillStyle = "#2CA02C"; c.fill(); c.lineWidth = 1.5; c.strokeStyle = "#fff"; c.stroke();
  }
  for (const e of CFG.fp) {                                // red cross at the predicted boundary
    if (e < vs || e > ve) continue;
    const x = tx(e), d = r * 0.9;
    c.lineCap = "round";
    c.strokeStyle = "#fff"; c.lineWidth = 5;
    c.beginPath(); c.moveTo(x - d, my - d); c.lineTo(x + d, my + d); c.moveTo(x - d, my + d); c.lineTo(x + d, my - d); c.stroke();
    c.strokeStyle = "#D62728"; c.lineWidth = 2.5;
    c.beginPath(); c.moveTo(x - d, my - d); c.lineTo(x + d, my + d); c.moveTo(x - d, my + d); c.lineTo(x + d, my - d); c.stroke();
    c.lineCap = "butt";
  }
  for (const g of CFG.fn) {                                // orange triangle at the missed GT boundary
    if (g < vs || g > ve) continue;
    const x = tx(g), d = r * 1.15;
    c.beginPath(); c.moveTo(x - d, my - d * 0.85); c.lineTo(x + d, my - d * 0.85); c.lineTo(x, my + d); c.closePath();
    c.fillStyle = "#FF7F0E"; c.fill(); c.lineWidth = 1; c.strokeStyle = "#222"; c.stroke();
  }
  c.restore();

  // ---- 7) time axis
  const steps = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100];
  const step = steps.find(s => s / span * plotW >= 80) || steps[steps.length - 1];
  const dec = step >= 1 ? 0 : step >= 0.1 ? 1 : step >= 0.01 ? 2 : 3;
  const first = Math.ceil(vs / step - 1e-9) * step;
  c.strokeStyle = "#555"; c.fillStyle = "#222"; c.font = "11px " + FONT; c.textAlign = "center"; c.textBaseline = "top";
  for (let i = 0; first + i * step <= ve + 1e-9; i++) {
    const t = first + i * step, x = Math.round(tx(t)) + 0.5;
    c.beginPath(); c.moveTo(x, axisY); c.lineTo(x, axisY + 5); c.stroke();
    c.fillText(t.toFixed(dec), x, axisY + 7);
  }
  c.textAlign = "right"; c.fillStyle = "#555"; c.fillText("Time (s)", L - 18, axisY + 7);

  updateCounts();
}

// ------------------------------------------------------------------ dynamic layer (playhead)
function render() {
  if (!geom) return;
  const { dpr, specY, predY, barH } = geom;
  ctx.setTransform(1, 0, 0, 1, 0, 0);
  ctx.clearRect(0, 0, cv.width, cv.height);
  ctx.drawImage(stat, 0, 0);
  if (sel) {                                               // highlight the interval that was clicked
    ctx.setTransform(dpr, 0, 0, dpr, 0, 0);
    const xa = Math.max(L, tx(sel.s)), xb = Math.min(L + geom.plotW, tx(sel.e));
    if (xb > xa) {
      ctx.fillStyle = "rgba(255,200,0,0.35)"; ctx.fillRect(xa, sel.y, xb - xa, sel.h);
      ctx.strokeStyle = "#111"; ctx.lineWidth = 2; ctx.strokeRect(xa + 1, sel.y + 1, Math.max(0, xb - xa - 2), sel.h - 2);
    }
  }
  if (playPos !== null && playPos >= vs && playPos <= ve) {
    ctx.setTransform(dpr, 0, 0, dpr, 0, 0);
    const x = tx(playPos);
    ctx.beginPath(); ctx.moveTo(x, specY); ctx.lineTo(x, predY + barH);
    ctx.strokeStyle = "rgba(255,255,255,0.85)"; ctx.lineWidth = 3; ctx.stroke();   // thin halo so it shows on dark spectrogram
    ctx.strokeStyle = "#000"; ctx.lineWidth = 1; ctx.stroke();
  }
}

// ------------------------------------------------------------------ view / controls
function updateCounts() {
  let tp = 0, fp = 0, fn = 0;
  for (const [g, e] of CFG.tp) if (e >= vs && e <= ve) tp++;
  for (const e of CFG.fp) if (e >= vs && e <= ve) fp++;
  for (const g of CFG.fn) if (g >= vs && g <= ve) fn++;
  $(".counts").innerHTML = "In view: <b style='color:#2CA02C'>TP " + tp + "</b> &nbsp;<b style='color:#D62728'>FP " + fp +
                           "</b> &nbsp;<b style='color:#FF7F0E'>FN " + fn + "</b>";
}

function syncInputs() {
  $(".start").value = vs.toFixed(2); $(".end").value = ve.toFixed(2);
  const w = ve - vs;
  $(".win").textContent = w.toFixed(2);
  const sl = $(".scroll"); sl.max = Math.max(0, dur - w); sl.value = vs;
}

function setView(s, e) {
  if (!isFinite(s) || !isFinite(e)) { syncInputs(); return; }
  s = Math.max(0, s); e = Math.min(dur, e);
  if (e - s < MIN_WIN) {
    if (s + MIN_WIN <= dur) e = s + MIN_WIN; else { e = dur; s = Math.max(0, dur - MIN_WIN); }
  }
  vs = s; ve = e;
  stopPlayback(false);
  syncInputs(); drawStatic(); render();
}

$(".start").addEventListener("change", ev => setView(parseFloat(ev.target.value), ve));
$(".end").addEventListener("change", ev => setView(vs, parseFloat(ev.target.value)));
$(".scroll").addEventListener("input", ev => { const w = ve - vs, v = parseFloat(ev.target.value); setView(v, v + w); });
$(".prev").addEventListener("click", () => { const w = ve - vs, s = Math.max(0, vs - w / 2); setView(s, s + w); });
$(".next").addEventListener("click", () => { const w = ve - vs, s = Math.min(dur - w, vs + w / 2); setView(s, s + w); });

// ------------------------------------------------------------------ playback
const btnPlay = $(".play");
function setTime(t) { $(".tnow").textContent = t === null ? "–" : t.toFixed(3) + " s"; }

function startPlayback(from, opts) {
  opts = opts || {};
  playEnd = opts.end !== undefined ? Math.min(opts.end, dur) : ve;        // default: end of the visible section
  loopFrom = opts.loopFrom !== undefined ? opts.loopFrom : vs;
  sel = opts.sel || null;
  playPos = from; audio.currentTime = from; setTime(from);
  const p = audio.play();
  if (p && p.catch) p.catch(err => { playing = false; btnPlay.innerHTML = "&#9654; Play"; console.warn("Audio could not start:", err); });
  playing = true; btnPlay.innerHTML = "&#10074;&#10074; Pause";
  cancelAnimationFrame(raf); raf = requestAnimationFrame(tick);
}
function pausePlayback() {
  audio.pause(); playing = false; cancelAnimationFrame(raf);
  btnPlay.innerHTML = "&#9654; Play"; render();
}
function stopPlayback(redraw) {
  audio.pause(); playing = false; playPos = null; sel = null; cancelAnimationFrame(raf);
  btnPlay.innerHTML = "&#9654; Play"; setTime(null);
  if (redraw !== false) render();
}
function tick() {
  if (!playing) return;
  const t = audio.currentTime;
  if (t >= playEnd - 0.005 || audio.ended) {
    if ($(".loop").checked) { startPlayback(loopFrom, { end: playEnd, loopFrom: loopFrom, sel: sel }); return; }
    stopPlayback(); return;
  }
  playPos = t; setTime(t); render();
  raf = requestAnimationFrame(tick);
}
btnPlay.addEventListener("click", () => {
  if (playing) { pausePlayback(); return; }
  const resume = playPos !== null && playPos < playEnd - 0.02;      // paused part-way: carry on (same section / same syllable)
  if (resume) startPlayback(playPos, { end: playEnd, loopFrom: loopFrom, sel: sel });
  else startPlayback(vs);
});
$(".stop").addEventListener("click", () => stopPlayback());

// Click a labelled interval (GT words / phones / syllables, or the prediction) to play just that interval.
// Click the spectrogram to play from that point to the end of the visible section.
cv.addEventListener("click", ev => {
  const rect = cv.getBoundingClientRect(), x = ev.clientX - rect.left, y = ev.clientY - rect.top;
  if (x < L || x > L + geom.plotW) return;
  const t = vs + (x - L) / geom.plotW * (ve - vs);
  const { specY, gtY, predY, barH } = geom, tierH = barH / 3;
  let ivs = null, row = null;
  if (y >= gtY && y < gtY + barH) {
    const k = Math.min(2, Math.floor((y - gtY) / tierH));
    ivs = [CFG.gt.words, CFG.gt.phones, CFG.gt.syllables][k]; row = { y: gtY + k * tierH, h: tierH };
  } else if (y >= predY && y < predY + barH) {
    ivs = CFG.est; row = { y: predY, h: barH };
  } else if (y >= specY && y < specY + barH) {
    startPlayback(t); return;
  } else return;
  const iv = ivs.find(it => t >= it[0] && t < it[1]);
  if (!iv || !iv[2]) return;                               // silences (empty intervals) are not played
  startPlayback(iv[0], { end: iv[1], loopFrom: iv[0], sel: { s: iv[0], e: iv[1], y: row.y, h: row.h } });
});
audio.addEventListener("ended", () => { if (playing && !$(".loop").checked) stopPlayback(); });

// ------------------------------------------------------------------ start up
$(".overall").textContent = "Overall (±" + Math.round(tol * 1000) + " ms): P " + CFG.metrics.precision.toFixed(3) +
  "  R " + CFG.metrics.recall.toFixed(3) + "  F1 " + CFG.metrics.f1.toFixed(3);
syncInputs(); setTime(null);
layout();
window.addEventListener("resize", layout);
if (window.ResizeObserver) new ResizeObserver(() => { if (geom && Math.abs(root.clientWidth - 18 - geom.W) > 2) layout(); }).observe(root);
})();
</script>
"""


def _round_ivs(tier):
    return [[round(float(iv.start_time), 4), round(float(iv.end_time), 4), iv.text] for iv in tier.intervals]


def build_viewer_html(height_px=None):
    payload = dict(
        duration=float(recording_duration),
        tolerance=float(TOLERANCE_S),
        initialStart=float(INITIAL_START_S),
        initialEnd=float(INITIAL_END_S),
        heightPx=height_px,
        gt=dict(words=_round_ivs(gt_word_tier), phones=_round_ivs(gt_phone_tier), syllables=_round_ivs(gt_tier)),
        est=_round_ivs(est_tier),
        scoredGT=[round(float(t), 4) for t in scored_gt_boundaries],
        tp=[[round(float(r), 4), round(float(e), 4)] for r, e in tp_pairs],
        fp=[round(float(t), 4) for t in fp_times],
        fn=[round(float(t), 4) for t in fn_times],
        spec=spec_info,
        metrics=dict(precision=float(precision), recall=float(recall), f1=float(f1)),
    )
    audio_b64 = base64.b64encode(Path(AUDIO_WAV).read_bytes()).decode()
    return (
        VIEWER_TEMPLATE
        .replace("__PAYLOAD__", json.dumps(payload, ensure_ascii=False).replace("</", "<\\/"))
        .replace("__AUDIO_B64__", audio_b64)
        .replace("__SPEC_URI__", spectrogram_uri)
        .replace("__UID__", uuid.uuid4().hex[:8])
    )


viewer_html = build_viewer_html(WIDGET_HEIGHT_PX)
display(HTML(viewer_html))

## 7. Optional: open the viewer as its own full-window page

If the notebook output area is too cramped (or your notebook is not trusted, which blocks the audio/playhead script), this writes the same viewer to a standalone HTML file. Double-click it to open in your browser, where it automatically fills the window.

In [ ]:
STANDALONE_HTML = Path("zerosyl_boundary_viewer.html")
STANDALONE_HTML.write_text(
    "<!doctype html><html><head><meta charset='utf-8'><title>ZeroSyl Boundary Evaluation 2</title></head>"
    "<body style='margin:8px;background:#fff'>" + build_viewer_html(None) + "</body></html>",
    encoding="utf-8",
)
print("Saved", STANDALONE_HTML.resolve())